# Tổng quan Milestone 1: Thu thập Dữ liệu Đa Phiên bản (25.000 Bài báo)

## 1. Mục tiêu học thuật và kiến trúc thu thập 4 pha gối đầu

Milestone 1 xây dựng hệ thống thu thập bền vững cho toàn bộ **25.000 arXiv ID** của 5 thành viên (`5 × 5.000` bài báo trải rộng từ `06/2024` đến `02/2025`), tuân thủ nghiêm ngặt hai đơn vị thống kê độc lập của đề bài:
- **Cấp bài báo (`publication`)**: đơn vị lưu trữ metadata tổng hợp (`title`, `authors`, `submission_date`, `revised_dates`, `venue`) và mạng lưới tài liệu tham khảo từ Semantic Scholar.
- **Cấp phiên bản (`(paper, version)`)**: đơn vị tải gói nguồn `/src/<id>vN`, lọc bỏ tệp nhị phân ngay trên luồng giải nén và kiểm toán độc lập toàn bộ lịch sử `v1..vN`.

```{note}
**Hoàn tất 100% dải bài báo trên Google Colab CPU-only trước khi bước sang M2:**
Thành viên `TRI (24120470)` ở lượt chạy đầu dừng tại `4.425 / 5.000` bài do chạm ngưỡng ngân sách phiên 11,5 giờ của Colab. Nhóm đã thực thi phiên `--resume` hoàn toàn trên **Google Colab CPU-only**, thu thập trọn vẹn `575` bài còn lại cùng `69` bài chờ xử lý, đạt **`5.000 / 5.000` bài** và xác minh **`8.338 / 8.338` phiên bản nguồn** trước khi hợp nhất toàn nhóm.
```

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch
from matplotlib.ticker import FuncFormatter
from IPython.display import display, Markdown, HTML

data = json.loads((Path.cwd().parent / "report_summary.json").read_text(encoding="utf-8"))
scope, m1, m2, bench = data["scope"], data["m1"], data["m2"], data["m2_benchmark"]

BOOK_PALETTE = {
    "navy": "#0F4C81",
    "teal": "#0D9488",
    "emerald": "#059669",
    "amber": "#D97706",
    "crimson": "#B31B1B",
    "slate": "#64748B",
    "sky": "#0284C7",
    "grid": "#CBD5E1",
}
MEMBER_COLORS = {
    "DAT": "#0F4C81",
    "TUAN": "#0D9488",
    "TAN": "#D97706",
    "TRI": "#059669",
    "TRIET": "#B31B1B",
}

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "figure.figsize": (9.8, 4.3),
    "figure.dpi": 180,
    "savefig.dpi": 180,
    "savefig.bbox": "tight",
    "figure.facecolor": "#FFFFFF",
    "axes.facecolor": "#FFFFFF",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": "#94A3B8",
    "axes.linewidth": 0.85,
    "axes.titlesize": 11.0,
    "axes.titleweight": "bold",
    "axes.titlecolor": "#0F4C81",
    "axes.labelsize": 9.5,
    "axes.labelcolor": "#1E293B",
    "xtick.labelsize": 8.9,
    "ytick.labelsize": 8.9,
    "xtick.color": "#334155",
    "ytick.color": "#334155",
    "grid.color": "#CBD5E1",
    "grid.linestyle": "--",
    "grid.alpha": 0.42,
})


def fmt_int(x) -> str:
    if x is None or pd.isna(x):
        return "—"
    return f"{int(round(float(x))):,}".replace(",", ".")


def fmt_num(x, digits: int = 2) -> str:
    if x is None or pd.isna(x):
        return "—"
    return f"{float(x):,.{digits}f}".replace(",", "X").replace(".", ",").replace("X", ".")


def fmt_pct(x, digits: int = 2, scale100: bool = False) -> str:
    if x is None or pd.isna(x):
        return "—"
    val = float(x) * (100.0 if scale100 else 1.0)
    return f"{val:.{digits}f}%".replace(".", ",")


def style_table(
    df: pd.DataFrame,
    caption: str | None = None,
    bar_cols: list[str] | dict[str, tuple[str, float | None]] | None = None,
    precision: int = 2,
    int_cols: list[str] | None = None,
    pct_cols: list[str] | None = None,
    pct_scale100: bool = False,
    col_formats: dict[str, object] | None = None,
    bar_color: str = "var(--ah-bar-fill)",
    vmax: float | None = None,
):
    out = df.copy()
    int_set = set(int_cols or [])
    pct_set = set(pct_cols or [])
    custom_fmt = dict(col_formats or {})

    for c in int_set:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors="coerce").round().astype("Int64")

    fmt_map: dict[str, object] = {}
    for col in out.columns:
        if col in custom_fmt:
            fmt_map[col] = custom_fmt[col]
        elif col in int_set or pd.api.types.is_integer_dtype(out[col]):
            fmt_map[col] = fmt_int
        elif col in pct_set:
            fmt_map[col] = lambda v, d=precision, s=pct_scale100: fmt_pct(v, d, s)
        elif pd.api.types.is_float_dtype(out[col]):
            fmt_map[col] = lambda v, d=precision: fmt_num(v, d)

    styler = out.style.hide(axis="index").set_table_attributes('class="dataframe ah-styled-table"')
    if caption:
        styler = styler.set_caption(caption)
    if fmt_map:
        styler = styler.format(fmt_map)

    if isinstance(bar_cols, dict):
        bar_spec = bar_cols
    else:
        bar_spec = {c: (bar_color, vmax) for c in (bar_cols or [])}

    for bc, (bcol, bmax) in bar_spec.items():
        if bc in out.columns:
            num_s = pd.to_numeric(out[bc], errors="coerce")
            hi = float(bmax) if bmax is not None else float(num_s.max() or 1.0)
            styler = styler.bar(subset=[bc], color=bcol, vmin=0.0, vmax=max(hi, 1e-9))

    num_cols = [
        c for c in out.columns
        if c in int_set or c in pct_set or c in bar_spec or pd.api.types.is_numeric_dtype(out[c])
    ]
    if num_cols:
        styler = styler.set_properties(
            subset=num_cols,
            **{"text-align": "right", "font-variant-numeric": "tabular-nums"},
        )
    return styler


print(
    f"Toàn nhóm (5 thành viên): {fmt_int(scope['papers'])} bài báo / "
    f"{fmt_int(scope['versions'])} phiên bản | Cập nhật: {data['report_date']}"
)

In [ ]:
# Sơ đồ kiến trúc Pipeline M1 4 pha chạy gối đầu (sinh trực tiếp lúc biên dịch)
fig, ax = plt.subplots(figsize=(10.2, 2.25))
ax.set_xlim(0, 10.2); ax.set_ylim(0, 2.25); ax.axis("off")
stages = [
    ("Pha 1 • Discovery", "Atom XML lô 50-100 ID\n+ OAI-PMH oaipmh.arxiv.org\n25.000 bài • 41.498 ver", "#0F4C81"),
    ("Pha 2 • Source Stream", "export.arxiv.org/src/<id>vN\nKhóa ≥3,0s + HTTP Range\nLọc tar -89,30% dung lượng", "#0284C7"),
    ("Pha 3 • S2 References", "POST /paper/batch (50/lô)\n+ Phân trang limit=1000\n976.351 dòng (0 mồ côi)", "#0D9488"),
    ("Pha 4 • Audit & Export", "1.713 gói backup SHA-256\nMerge khử trùng 5 thành viên\npapers + references (6+6 cột)", "#D97706"),
]
for idx, (st_title, st_desc, col) in enumerate(stages):
    x0 = 0.15 + idx * 2.52
    ax.add_patch(FancyBboxPatch((x0, 0.22), 2.22, 1.72, boxstyle="round,pad=0.06,rounding_size=0.10",
                                facecolor="#F8FAFC", edgecolor=col, linewidth=1.5))
    ax.add_patch(FancyBboxPatch((x0, 1.48), 2.22, 0.46, boxstyle="round,pad=0.04,rounding_size=0.08",
                                facecolor=col, edgecolor=col, linewidth=1.0))
    ax.text(x0 + 1.11, 1.71, st_title, ha="center", va="center", color="white", fontsize=9.2, fontweight="bold")
    ax.text(x0 + 1.11, 0.84, st_desc, ha="center", va="center", color="#1E293B", fontsize=8.2, linespacing=1.35)
    if idx < 3:
        ax.add_patch(FancyArrowPatch((x0 + 2.28, 1.08), (x0 + 2.50, 1.08),
                                     arrowstyle="-|>,head_length=6,head_width=4", color="#64748B", lw=1.6))
ax.set_title("Hình M1.0. Kiến trúc thu thập đa luồng gối đầu của Milestone 1 (tách biệt khóa nhịp arXiv ≥ 3,0s và Semantic Scholar)", pad=4)
plt.tight_layout(); plt.show()

In [ ]:
df_m1_1 = pd.DataFrame([
    ("Tổng arXiv ID được phân công (5 thành viên)", scope["group_assigned"], "bài báo"),
    ("Số bài báo hợp nhất duy nhất (0 trùng lặp chéo)", scope["papers"], "bài báo"),
    ("Tổng số phiên bản ghi nhận (v1..vN)", scope["versions"], "phiên bản"),
    ("Chuỗi gói sao lưu gia tăng kiểm chứng SHA-256", scope["archives_used"], "gói archive"),
    ("Tổng dòng trích dẫn tầng Working (0 mồ côi)", m1["working_reference_rows"], "cạnh trích dẫn"),
    ("Cạnh trích dẫn chuẩn môn học (có đích ref_arxiv_id)", m1["arxiv_reference_rows"], "cạnh trích dẫn"),
], columns=["Chỉ tiêu tổng hợp", "Số lượng", "Đơn vị"])
display(style_table(df_m1_1, caption="Bảng M1.1. Quy mô kho dữ liệu hợp nhất 5 thành viên sau khử trùng.", int_cols=["Số lượng"]))

In [ ]:
sa = m1["source_audit"]
metrics = pd.DataFrame([
    ("Phiên bản có mã nguồn TeX xác minh (tex_verified)", sa.get("tex_verified", 0), scope["versions"]),
    ("Phiên bản đã kiểm toán hợp lệ (gồm PDF-only / no-LaTeX)", m1["source_verified"], scope["versions"]),
    ("Bài báo xác minh hợp lệ đủ 100% phiên bản", m1["papers_with_all_sources_verified"], scope["papers"]),
    ("Bài báo có phản hồi S2 xác minh (success / empty)", m1["verified_reference_papers"], scope["papers"]),
    ("Bài báo hoàn tất trọn vẹn cả nguồn lẫn trích dẫn S2", m1["papers_with_sources_and_s2_verified"], scope["papers"]),
], columns=["Chỉ tiêu độ phủ", "Đạt", "Mẫu số"])
metrics["Tỷ lệ (%)"] = 100.0 * metrics["Đạt"] / metrics["Mẫu số"]
display(style_table(
    metrics,
    caption="Bảng M1.2. Độ phủ kiểm toán nguồn và trích dẫn trên toàn bộ 25.000 bài báo.",
    int_cols=["Đạt", "Mẫu số"],
    pct_cols=["Tỷ lệ (%)"],
    bar_cols={"Tỷ lệ (%)": ("rgba(13, 148, 136, 0.22)", 100.0)},
))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.8, 4.1), gridspec_kw={"width_ratios": [1.35, 1.0]})
y_pos = np.arange(len(metrics))
bars = ax1.barh(y_pos, metrics["Tỷ lệ (%)"], color=["#0284C7", "#0F4C81", "#059669", "#0D9488", "#D97706"], height=0.58)
ax1.set_yticks(y_pos); ax1.set_yticklabels(metrics["Chỉ tiêu độ phủ"], fontsize=8.6)
ax1.invert_yaxis(); ax1.set_xlim(0, 118); ax1.grid(axis="x")
ax1.set_xlabel("Tỷ lệ đạt (%)"); ax1.set_title("Hình M1.1a. Năm chỉ tiêu độ phủ M1")
for b, r in zip(bars, metrics.to_dict("records")):
    ax1.text(r["Tỷ lệ (%)"] + 1.2, b.get_y() + b.get_height()/2,
             f"{fmt_int(r['Đạt'])} ({fmt_pct(r['Tỷ lệ (%)'])})", va="center", fontsize=8.2, fontweight="bold")

ssb = pd.Series(m1["source_status_breakdown"]).sort_values(ascending=True)
ssb_lbl = {
    "success": "Đủ TeX mọi version",
    "pdf_only": "Chỉ nộp PDF (pdf_only)",
    "partial_success": "Có TeX một phần",
    "not_found": "HTTP 404 trên arXiv",
    "no_latex": "Gói nguồn không có TeX",
    "error": "Chờ/lỗi tải nguồn",
}
ax2.barh([ssb_lbl.get(k, k) for k in ssb.index], ssb.values, color="#0F4C81", height=0.58)
ax2.set_xscale("symlog", linthresh=50)
ax2.set_xlim(0, 80000); ax2.grid(axis="x")
ax2.set_xlabel("Số bài báo (thang symlog)"); ax2.set_title("Hình M1.1b. Phân loại cấp bài báo (25.000 bài)")
for i, v in enumerate(ssb.values):
    ax2.text(v * 1.15 + 2, i, fmt_int(v), va="center", fontsize=8.2, fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
pm1 = pd.DataFrame(m1["per_member"])
view_m1 = pd.DataFrame({
    "Thành viên": pm1["member"],
    "Dải arXiv ID": pm1["id_range"],
    "Số bài": pm1["publications"],
    "Phiên bản": pm1["versions"],
    "TeX xác minh": pm1["tex_verified"],
    "PDF/Empty": pm1["empty_verified"],
    "Working Refs": pm1["working_references"],
    "Course Refs (arXiv)": pm1["course_references"],
    "Giảm lọc ảnh (%)": 100.0 * pm1["source_reduction_rate"],
})
display(style_table(
    view_m1,
    caption="Bảng M1.3. Phân rã kết quả thu thập Milestone 1 theo 5 thành viên (mỗi phân vùng 5.000 bài, 0 trùng chéo).",
    int_cols=["Số bài", "Phiên bản", "TeX xác minh", "PDF/Empty", "Working Refs", "Course Refs (arXiv)"],
    pct_cols=["Giảm lọc ảnh (%)"],
    bar_cols={"Course Refs (arXiv)": ("var(--ah-bar-fill)", None)},
))

## Tài liệu và mã nguồn tham chiếu

- [Đặc tả chính thức Milestone 1 — Data Crawling](https://hcmus-cs-ta.github.io/lab-book/intro-ds/statement/crawling/).
- [arXiv API User's Manual](https://info.arxiv.org/help/api/user-manual.html), [API Terms of Use](https://info.arxiv.org/help/api/tou.html) & [OAI-PMH Cutover (`oaipmh.arxiv.org`)](https://info.arxiv.org/help/oa/index.html).
- [Semantic Scholar Academic Graph API](https://api.semanticscholar.org/api-docs/graph).
- Mã nguồn triển khai: [`src/crawler/main.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/src/crawler/main.py).
- Mã nguồn triển khai: [`src/crawler/merge_shards.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/src/crawler/merge_shards.py).
- Mã nguồn triển khai: [`src/crawler/validate_sources.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/src/crawler/validate_sources.py).